# 01 — Preprocessing for LLM pipelines (Part A)

**Goal:** build the building blocks of a conservative cleaning pipeline (🧩 exercises), package them in `src/preprocessing.py` (tests), then *prove with experiments* why a conservative pipeline is the right default for LLM data.

Your name(s): _..._

**How the 🧩 exercises work.** Cells marked 🧩 contain `____` blanks: complete them using the hints (they name the Python functions you need). Each exercise is followed by a ✅ *check* cell that tells you immediately whether you got it right. When an exercise matches a function of `src/`, the docstring of that function says so: you then **move your working code into `src/`** and run `pytest`.

In [ ]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd().parent          # notebooks/ is one level below the repo root
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

from src.preprocessing import *
import unicodedata, re

## A.1 Theory

Answer in your own words (2–5 sentences each). Cite your sources in the report.

1. Data cleaning vs. text normalization vs. tokenization?
2. Why is stop-word removal inappropriate for LLM pretraining?
3. Why preserve punctuation and capitalization?
4. What is Unicode? Unicode vs. UTF-8?
5. What is Unicode normalization? NFC vs. NFD?
6. Why does deduplication matter for an LLM corpus?
7. Which problems appear when preprocessing multilingual text?
8. Why can aggressive normalization damage the information in a sentence?

**Your answer:**

_(write here)_

## Building block 1 — Unicode

### 🧩 Exercise 1.1 — Unicode, code points and bytes

The same visible text can be stored in different ways. Normalize the decomposed form to NFC, compare it with the composed form, count its UTF-8 bytes, and print the Unicode names of the code points of the decomposed form.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `unicodedata.normalize(form, text)` — `form` is a *string*: `"NFC"`, `"NFD"`, `"NFKC"`, `"NFKD"`.
* `text.encode("utf-8")` returns the bytes; `len(...)` counts them.
* `unicodedata.name(char)` gives the official name of one character.
* Docs: https://docs.python.org/3/library/unicodedata.html

In [ ]:
nfd = "e\u0301"      # 'e' followed by a combining acute accent
nfc = "\u00e9"       # precomposed 'é'
print(nfd == nfc, len(nfd), len(nfc))

normalized = unicodedata.____("____", nfd)
same_after = normalized == nfc
n_bytes = len(nfc.____("utf-8"))
names = [unicodedata.____(c) for c in nfd]
print(same_after, n_bytes, names)

In [ ]:
assert same_after is True and n_bytes == 2 and len(names) == 2
assert unicodedata.normalize("NFD", nfc) == nfd
print("✅ Exercise 1.1 OK")

In [ ]:
# NFC vs NFKC on a ligature and on Arabic: look at what NFKC destroys
for s in ["\ufb01nance", "ﻻ", "السَّلام"]:
    print(repr(s), "NFC ->", repr(unicodedata.normalize("NFC", s)), "| NFKC ->", repr(unicodedata.normalize("NFKC", s)))

**Question:** why does the assignment require **NFC** and not **NFKC**? Use the output above.

**Your answer:**

_(write here)_

## Building block 2 — Whitespace

### 🧩 Exercise 1.2 — Normalize whitespace with a regular expression

Collapse every run of whitespace (spaces, tabs, newlines, non-breaking spaces) into a single space, and strip both ends.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `re.sub(pattern, replacement, text)` replaces every match.
* In a Python `str` regex, `\s+` matches one or more whitespace characters (including `\xa0`).
* `str.strip()` removes leading/trailing whitespace.

In [ ]:
messy = "  Hello \t  world\n\n \u00a0 !  "
clean = re.____(r"____", ____, messy).____()
print(repr(clean))

In [ ]:
assert clean == "Hello world !"
print("✅ Exercise 1.2 OK")

## Building block 3 — HTML to text

A regex such as `re.sub(r"<[^>]+>", " ", html)` fails on `This is <b>very</b>.` (it leaves `very .`) and on `5 &lt; 10`. Python's standard library has a real parser: `html.parser.HTMLParser`. Complete the class below: block tags (`<p>`, `<div>`, `<br>`…) separate text with a space, **inline** tags (`<b>`, `<i>`, `<a>`) do not, and the content of `<script>`/`<style>` is dropped.

### 🧩 Exercise 1.3 — Complete an HTMLParser subclass

Fill the blanks of `TextExtractor`. The parser calls `handle_starttag`, `handle_endtag` and `handle_data` as it reads the document.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `HTMLParser(convert_charrefs=True)` decodes entities such as `&amp;` and `&lt;` for you (call the parent constructor with `super().__init__(...)`).
* Keep a counter `skip_depth`: +1 when a `<script>`/`<style>` starts, −1 when it ends; text is only kept while it is 0.
* Block tags append a single space `" "` to `self.parts` (at start **and** end); inline tags append nothing.
* `handle_data(data)` receives the text between tags: append it **only** if you are not inside a skipped tag.
* Docs: https://docs.python.org/3/library/html.parser.html

In [ ]:
from html.parser import HTMLParser

BLOCK_TAGS = {"p", "div", "br", "li", "ul", "ol", "tr", "td", "th", "table", "h1", "h2", "h3", "section", "article"}
SKIP_TAGS = {"script", "style"}

class TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__(____)
        self.parts = []
        self.skip_depth = 0

    def handle_starttag(self, tag, attrs):
        if tag in SKIP_TAGS:
            self.skip_depth ____
        elif tag in BLOCK_TAGS:
            self.parts.append(____)

    def handle_endtag(self, tag):
        if tag in SKIP_TAGS:
            self.skip_depth = max(0, self.skip_depth - 1)
        elif tag in ____:
            self.parts.append(" ")

    def handle_data(self, data):
        if self.skip_depth ____:
            self.parts.append(____)

def html_to_text(html):
    parser = TextExtractor()
    parser.feed(html)
    parser.close()
    return "".join(parser.parts)

for h in ["<p>Hello <b>world</b>.</p><p>Second</p>", "5 &lt; 10 &amp; 10 &gt; 5", "<script>var x=1;</script><div>Visible.</div>"]:
    print(repr(re.sub(r"\s+", " ", html_to_text(h)).strip()))

In [ ]:
squash = lambda s: re.sub(r"\s+", " ", s).strip()
assert squash(html_to_text("This is <b>very</b>. Really.")) == "This is very. Really."
assert squash(html_to_text("<p>First</p><p>Second</p>")) == "First Second"
assert squash(html_to_text("5 &lt; 10 &amp; 10 &gt; 5")) == "5 < 10 & 10 > 5"
assert squash(html_to_text("<script>var x=1;</script><style>p{}</style><div>Visible.</div>")) == "Visible."
print("✅ Exercise 1.3 OK")

## Building block 4 — Exact deduplication that keeps the order

### 🧩 Exercise 1.4 — Remove duplicates, keep the first occurrence and the order

Complete `dedupe`. (`list(set(docs))` loses the order: do it with a loop.)

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* A `set` remembers what you have seen and tests membership in O(1): `x in seen`, `seen.add(x)`.
* Create an empty set with `set()` (the literal `{}` is an empty *dict*).
* Append a document to `unique` only the first time you see it.

In [ ]:
def dedupe(docs):
    seen = ____
    unique = []
    for d in docs:
        if d ____ seen:
            seen.____(d)
            unique.append(d)
    return unique

print(dedupe(["b", "a", "b", "c", "a"]))

In [ ]:
assert dedupe(["b", "a", "b", "c", "a"]) == ["b", "a", "c"]
assert dedupe(["Apple", "apple", "Apple"]) == ["Apple", "apple"]
assert dedupe([]) == []
print("✅ Exercise 1.4 OK")

## ⏸ Stop: package your code in `src/preprocessing.py`

| You wrote in… | → package it in |
|---|---|
| exercise 1.1 | `normalize_unicode` |
| exercise 1.2 | `normalize_whitespace` |
| exercise 1.3 | `remove_html` (put the class in `src/preprocessing.py`) |
| exercise 1.4 | `remove_duplicates` |

Then write `preprocess_text`, `drop_empty` and `preprocess_corpus`, and run `pytest tests/test_preprocessing.py`. When it is green, continue.

## A.2 Run your pipeline on the messy dataset

`data/raw/messy_documents.jsonl`: HTML, missing values, whitespace noise, NFD/NFC variants, French, Arabic, Darija, and duplicates that only appear *after* cleaning.

In [ ]:
with open(ROOT / "data/raw/messy_documents.jsonl", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]
raw_docs = [r["text"] for r in records]
print(len(raw_docs), "raw records")
pd.DataFrame({"raw": [repr(d) for d in raw_docs], "clean": [preprocess_text(d) for d in raw_docs]})

In [ ]:
clean_docs, stats = preprocess_corpus(raw_docs)
print(stats)
for d in clean_docs: print("-", d)

**Questions**
- Which records were removed as empty, which as duplicates? Why were the NFD/NFC French sentences detected as duplicates only *after* normalization?
- Explain the order of your pipeline (HTML → Unicode → whitespace). Give an example where another order gives a wrong result.
- Which design decisions did you make that the tests do not cover? (newlines, zero-width characters…)

**Your answer:**

_(write here)_

## A.3 Experiment — conservative vs. aggressive

In [ ]:
sentences = [
    "The model is NOT performing well!", "The model is not performing well.",
    "Let's eat, grandma!", "Let's eat grandma!",
    "Apple released a new product.", "I ate an apple yesterday.",
]
df = pd.DataFrame({
    "original": sentences,
    "aggressive": [aggressive_normalize(s) for s in sentences],
    "conservative": [preprocess_text(s) for s in sentences],
})
df

### 🧩 Exercise 1.5 — Find the collisions with pandas

A *collision* is a group of different original sentences that become the **same** string after a pipeline. Detect them for the aggressive output.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `df.groupby(column)[other_column].apply(list)` gives, for every distinct value of `column`, the list of originals.
* `series.map(len)` computes the size of each group; keep groups with more than one element: `groups[condition]`.
* Do it for `"aggressive"` first; then loop over both columns to compare.

In [ ]:
groups = df.____("aggressive")["original"].____(list)
collisions = groups[groups.____(len) ____]
for out, originals in collisions.items():
    print(repr(out), "<-", originals)

In [ ]:
assert len(collisions) >= 1
assert all(len(v) > 1 for v in collisions)
print("✅ Exercise 1.5 OK")

In [ ]:
for col in ["aggressive", "conservative"]:
    g = df.groupby(col)["original"].apply(list)
    print(f"--- {col}: {int((g.map(len) > 1).sum())} collision(s)")

**Discuss**
1. Which transformations lose information? (lowercasing / punctuation removal / stop-word removal, one by one)
2. Which pairs of sentences with different meanings became identical?
3. Why does preserving this information matter for **next-token prediction**?

**Your answer:**

_(write here)_

## Bonus (not graded): ordering bug

What happens if you apply `aggressive_normalize` to **NFD** text? What does it tell you about the order of the pipeline?

In [ ]:
print(aggressive_normalize("L'e\u0301le\u0300ve a\u0300 l'e\u0301cole"))
print(aggressive_normalize("L'\u00e9l\u00e8ve \u00e0 l'\u00e9cole"))
print(aggressive_normalize("السَّلامُ عَلَيْكُم"))   # what happened to the diacritics?